In [3]:
import pandas as pd
import numpy as np
from decimal import Decimal
import glob
import backtrader as bt
import matplotlib.pyplot as plt
import pandas_ta as ta

# Ensure plots are rendered inline in the notebook
%matplotlib inline

In [4]:
# Define the path to your CSV files
file_path_pattern = '/root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_*.csv'

# Define the column names
column_names = ['DATE', 'TIME', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'TICKVOL', 'VOL', 'SPREAD']

# Load and combine all CSV files
all_files = glob.glob(file_path_pattern)
data_list = []

for file in all_files:
    print(f"Loading file: {file}")  # Debugging: Print file being loaded
    df = pd.read_csv(file, delimiter=',', names=column_names, header=None, dtype=str)
    # print(df.head())  # Debugging: Print the first few rows of the loaded DataFrame
    data_list.append(df)

combined_data = pd.concat(data_list)

# Check the combined data before parsing dates
print("Combined data before parsing dates:")
print(combined_data.head())

# Combine <DATE> and <TIME> into a single datetime column
combined_data['datetime'] = pd.to_datetime(combined_data['DATE'] + ' ' + combined_data['TIME'], format='%Y.%m.%d %H:%M', errors='coerce')

# Check the combined data after parsing dates
print("Combined data after parsing dates:")
print(combined_data.head())

# Check for rows with NaT in datetime column
print("Rows with NaT in datetime column:")
print(combined_data[combined_data['datetime'].isna()].head())

# Drop rows with NaT in datetime column
combined_data.dropna(subset=['datetime'], inplace=True)

# Set datetime as the index
combined_data.set_index('datetime', inplace=True)
combined_data.sort_index(inplace=True)

# Check the combined data before converting numeric columns
print("Combined data before converting numeric columns:")
print(combined_data.head())

# Convert numeric columns to appropriate data types
numeric_columns = ['OPEN', 'HIGH', 'LOW', 'CLOSE', 'TICKVOL', 'VOL', 'SPREAD']
combined_data[numeric_columns] = combined_data[numeric_columns].apply(pd.to_numeric, errors='coerce')

# Check the combined data after converting numeric columns
print("Combined data after converting numeric columns:")
print(combined_data.head())

# Drop rows with NaN values in numeric columns
combined_data.replace([np.inf, -np.inf], np.nan, inplace=True)
combined_data.fillna(0, inplace=True)

# Display the final combined data
print("Final combined data:")
combined_data.head()

Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2021.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2018.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2002.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2000.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2017.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2001.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2003.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2013.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2020.csv
L

,DATE,TIME,OPEN,HIGH,LOW,CLOSE,TICKVOL,VOL,SPREAD
datetime,,,,,,,,,
2000-05-30 17:27:00,2000.05.30,17:27,0.9302,0.9302,0.9302,0.9302,0,0.0,0.0
2000-05-30 17:35:00,2000.05.30,17:35,0.9304,0.9305,0.9304,0.9305,0,0.0,0.0
2000-05-30 17:38:00,2000.05.30,17:38,0.9304,0.9304,0.9303,0.9303,0,0.0,0.0
2000-05-30 17:43:00,2000.05.30,17:43,0.9301,0.9301,0.9300,0.9300,0,0.0,0.0
2000-05-30 17:44:00,2000.05.30,17:44,0.9298,0.9298,0.9297,0.9297,0,0.0,0.0


In [5]:


# Calculate indicators
combined_data['AROON-OSC'] = ta.aroon(combined_data['HIGH'], combined_data['LOW'], length=14)['AROONOSC_14']
combined_data['EMA_50'] = ta.ema(combined_data['CLOSE'], length=50)
combined_data['STOCH-RSId'] = ta.stochrsi(combined_data['CLOSE'], length=14, smoothK=3, smoothD=3)['STOCHRSId_14_14_3_3']
combined_data['STOCH-RSIk'] = ta.stochrsi(combined_data['CLOSE'], length=14, smoothK=3, smoothD=3)['STOCHRSIk_14_14_3_3']
combined_data['SUPERTREND'] = ta.supertrend(combined_data['HIGH'], combined_data['LOW'], combined_data['CLOSE'], length=50, multiplier=3.0)['SUPERT_50_3.0']


# Drop rows with NaN values (due to indicator calculation)
combined_data.dropna(inplace=True)

# Display the data with indicators
print("Data with indicators:")
combined_data

Data with indicators:


,DATE,TIME,OPEN,HIGH,LOW,CLOSE,TICKVOL,VOL,SPREAD,AROON-OSC,EMA_50,STOCH-RSId,STOCH-RSIk,SUPERTREND
datetime,,,,,,,,,,,,,,
2000-05-30 19:56:00,2000.05.30,19:56,0.92970,0.92970,0.92970,0.92970,0,0.0,0.0,71.428571,0.929773,80.492573,78.194824,0.929298
2000-05-30 19:57:00,2000.05.30,19:57,0.92960,0.92960,0.92950,0.92950,0,0.0,0.0,71.428571,0.929762,78.620108,79.470677,0.929298
2000-05-30 19:58:00,2000.05.30,19:58,0.92970,0.92990,0.92970,0.92980,0,0.0,0.0,85.714286,0.929764,81.343142,86.363924,0.929378
2000-05-30 19:59:00,2000.05.30,19:59,0.92990,0.93050,0.92990,0.93050,0,0.0,0.0,92.857143,0.929793,84.066175,86.363924,0.929745
2000-05-30 20:00:00,2000.05.30,20:00,0.93060,0.93070,0.93060,0.93070,0,0.0,0.0,100.000000,0.929828,90.909283,100.000000,0.930192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-01-31 23:53:00,2023.01.31,23:53,1.08656,1.08659,1.08655,1.08659,0,0.0,0.0,-50.000000,1.086529,38.277047,43.473151,1.086708
2023-01-31 23:54:00,2023.01.31,23:54,1.08660,1.08660,1.08655,1.08656,0,0.0,0.0,-50.000000,1.086530,43.498340,47.571018,1.086708
2023-01-31 23:55:00,2023.01.31,23:55,1.08656,1.08658,1.08655,1.08656,0,0.0,0.0,-50.000000,1.086532,47.571018,51.668886,1.086708


In [6]:
class SuperTrend(bt.Indicator):
    lines = ('supertrend',)
    params = (('period', 7), ('multiplier', 3))

    def __init__(self):
        atr = bt.indicators.ATR(self.data, period=self.params.period)
        hl2 = (self.data.high + self.data.low) / 2
        self.l.basic_ub = hl2 + (self.params.multiplier * atr)
        self.l.basic_lb = hl2 - (self.params.multiplier * atr)
        self.l.final_ub = self.l.basic_ub
        self.l.final_lb = self.l.basic_lb
        self.l.supertrend = self.l.final_ub

    def next(self):
        if self.data.close[-1] > self.l.final_ub[-1]:
            self.l.final_ub[0] = min(self.l.basic_ub[0], self.l.final_ub[-1])
        else:
            self.l.final_ub[0] = self.l.basic_ub[0]

        if self.data.close[-1] < self.l.final_lb[-1]:
            self.l.final_lb[0] = max(self.l.basic_lb[0], self.l.final_lb[-1])
        else:
            self.l.final_lb[0] = self.l.basic_lb[0]

        if self.data.close[0] > self.l.final_ub[0]:
            self.l.supertrend[0] = self.l.final_lb[0]
        else:
            self.l.supertrend[0] = self.l.final_ub[0]

In [25]:
class SuperTrendStrategy(bt.Strategy):
    params = dict(
        st_length=50,
        st_multiplier=3.0,
        ema_length=50,
        aroon_length=50,
        risk_per_trade=0.0025,
        rr_ratio=2.8,
    )

    def __init__(self):
        # Indicators
        self.supertrend = SuperTrend(
            self.data,
            period=self.params.st_length,
            multiplier=self.params.st_multiplier,
            plot=True
        )
        self.ema = bt.indicators.EMA(self.data.close, period=self.params.ema_length, plot=True)
        self.aroon = bt.indicators.AroonOscillator(self.data, period=self.params.aroon_length, plot=True)
        self.stoch_rsi = bt.indicators.Stochastic(self.data, plot=True)

    def log(self, *args, dt=None):
        '''Logging function for this strategy'''
        dt = dt or self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), ', '.join(map(str, args))))
    
    def calculate_take_profit(self, current_price, stop_loss, is_buy):
            if is_buy:
                return current_price + ((current_price - stop_loss) * self.params.rr_ratio)
            else:
                return current_price - ((stop_loss - current_price) * self.params.rr_ratio)

    def calculate_volume(self, current_price, stop_loss):
        # Calculate position size based on risk
        account_balance = self.broker.getvalue()
        risk_amount = account_balance * self.params.risk_per_trade
        stop_loss_distance = abs(current_price - stop_loss)

        # Ensure stop loss distance is not zero or too small
        if stop_loss_distance < 1e-5:
            print(f"Skipping trade due to small stop loss distance - Current Price: {current_price}, Stop Loss: {stop_loss}")
            return None

        # Calculate volume in lots (1 lot = 100,000 units)
        volume = risk_amount / stop_loss_distance / 100000
        return round(volume, 2)
            

    def next(self):
        # Fetch values
        current_price = self.data.close[0]
        supertrend = self.supertrend[0]
        ema = self.ema[0]
        aroon = self.aroon[0]
        stoch_k = self.stoch_rsi.percK[0]
        stoch_d = self.stoch_rsi.percD[0]

        # Log indicator values
        # print(f"Current Price: {current_price:.5f}, Supertrend: {supertrend:.5f}, EMA: {ema:.5f}, Aroon: {aroon:.2f}, Stoch K: {stoch_k:.2f}, Stoch D: {stoch_d:.2f}")

        # Calculate stop loss and take profit
        stop_loss = round(supertrend, 5)
        
        volume = self.calculate_volume(current_price, stop_loss)

        if volume is None or volume <= 0:
            print(f"Skipping trade due to small stop loss distance or invalid volume - Current Price: {current_price:.5f}, Stop Loss: {stop_loss:.5f}")
            return
        
        risk_value = abs(current_price - stop_loss) * volume * 100000
        
        
        # Long entry conditions
        if current_price > supertrend and current_price > ema and aroon > 60 and stoch_k < 30 and stoch_d < stoch_k:
            take_profit = round(self.calculate_take_profit(current_price, stop_loss, is_buy=True), 5)
            take_profit_value = abs(current_price - take_profit) * volume * 100000
            self.log(f"Buy Condition Met - Current Price: {current_price:.5f}, Supertrend: {supertrend:.5f}, EMA: {ema:.5f}, Aroon: {aroon:.2f}, Stoch K: {stoch_k:.2f}, Stoch D: {stoch_d:.2f}")
            self.log(f"Risk Value: {risk_value:.2f}, Take Profit Value: {take_profit_value:.2f}, Volume: {volume}")
            self.order = self.buy(size=volume)
            self.sell_order = self.sell(size=volume, exectype=bt.Order.Stop, price=stop_loss)
            self.sell_order = self.sell(size=volume, exectype=bt.Order.Limit, price=take_profit)
            self.log(f"SL: {stop_loss}, TP: {take_profit}, Volume: {volume}")
        
        # Short entry conditions
        if current_price < supertrend and current_price < ema and aroon < -60 and stoch_k > 70 and stoch_d > stoch_k:
            take_profit = round(self.calculate_take_profit(current_price, stop_loss, is_buy=False), 5)
            take_profit_value = abs(current_price - take_profit) * volume * 100000
            self.log(f"Sell Condition Met - Current Price: {current_price:.5f} Supertrend: {supertrend:.5f}, EMA: {ema:.5f}, Aroon: {aroon:.2f}, Stoch K: {stoch_k:.2f}, Stoch D: {stoch_d:.2f}")
            self.log(f"Risk Value: {risk_value:.2f}, Take Profit Value: {take_profit_value:.2f}, Volume: {volume}")
            self.order = self.sell(size=volume)
            self.buy_order = self.buy(size=volume, exectype=bt.Order.Stop, price=stop_loss)
            self.buy_order = self.buy(size=volume, exectype=bt.Order.Limit, price=take_profit)
            self.log(f"SL: {stop_loss}, TP: {take_profit}, Volume: {volume}")

    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            return

        if order.status in [order.Completed]:
            if order.isbuy() and order == self.buy_order:
                self.log(f"STOP LOSS HIT on BUY, Price: {order.executed.price:.5f}...")
            elif order.issell() and order == self.buy_order:
                self.log(f"STOP LOSS HIT on SELL, Price: {order.executed.price:.5f}...")

            self.bar_executed = len(self)

        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log('Order Canceled/Margin/Rejected')

        self.order = None

    def notify_trade(self, trade):
        if not trade.isclosed:
            return

        self.log(f'TRADE PROFIT, GROSS {trade.pnl:.2f}, NET {trade.pnlcomm:.2f}')
        self.log(f'New Account Balance: {self.broker.getvalue():.2f}')


In [26]:
# Convert the pandas DataFrame to a Backtrader data feed
class PandasData(bt.feeds.PandasData):
    lines = (
        'ema_50',
        'stoch_rsi_k',
        'stoch_rsi_d',
        'supertrend',
        'aroon_osc',
    )
    params = (
        ('datetime', None),
        ('open', 'OPEN'),
        ('high', 'HIGH'),
        ('low', 'LOW'),
        ('close', 'CLOSE'),
        ('volume', 'VOL'),
        ('ema_50', 'EMA_50'),
        ('stoch_rsi_k', 'STOCH-RSIk'),
        ('stoch_rsi_d', 'STOCH-RSId'),
        ('supertrend', 'SUPERTREND'),
        ('aroon_osc', 'AROON-OSC'),
    )

test_data = combined_data.iloc[-10000:]

data_feed = PandasData(dataname=test_data)

comm_info = bt.CommInfoBase(
    commission=0.02,
    leverage=30,
    margin=1 / 30,
    mult=100000
)

# Set up the Backtrader environment
cerebro = bt.Cerebro()
cerebro.addstrategy(SuperTrendStrategy)
cerebro.adddata(data_feed)
cerebro.broker.set_cash(2500)
cerebro.broker.addcommissioninfo(comm_info)

# Run the backtest
cerebro.run(style='candlestick')
print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())
# Plot the results
cerebro.plot()
plt.show()

2023-01-23, Sell Condition Met - Current Price: 1.08997 Supertrend: 1.09097, EMA: 1.08998, Aroon: -64.00, Stoch K: 72.92, Stoch D: 79.03
2023-01-23, Risk Value: 6.00, Take Profit Value: 16.80, Volume: 0.06
2023-01-23, SL: 1.09097, TP: 1.08717, Volume: 0.06
2023-01-23, TRADE PROFIT, GROSS -6.06, NET -6.06
2023-01-23, New Account Balance: 2493.94
2023-01-23, Sell Condition Met - Current Price: 1.09002 Supertrend: 1.09068, EMA: 1.09027, Aroon: -80.00, Stoch K: 75.59, Stoch D: 80.91
2023-01-23, Risk Value: 5.94, Take Profit Value: 16.65, Volume: 0.09
2023-01-23, SL: 1.09068, TP: 1.08817, Volume: 0.09
2023-01-23, Sell Condition Met - Current Price: 1.09013 Supertrend: 1.09087, EMA: 1.09026, Aroon: -80.00, Stoch K: 70.09, Stoch D: 73.45
2023-01-23, Risk Value: 5.92, Take Profit Value: 16.56, Volume: 0.08
2023-01-23, SL: 1.09087, TP: 1.08806, Volume: 0.08
2023-01-23, Sell Condition Met - Current Price: 1.09020 Supertrend: 1.09085, EMA: 1.09026, Aroon: -78.00, Stoch K: 71.33, Stoch D: 72.03
20

<IPython.core.display.Javascript object>